# Plan judge v2 on Colab (protocol 2, rank 1, one call per Plan)

1. Edit **only** the config cell below.
2. Run all cells. `MODE = "dry_run"` (default) validates inputs and shows the prompt without loading a model.
3. `MODE = "judge"` judges the selected variants. It resumes automatically; rerun the same config after a disconnect.
4. `MODE = "validate"` reruns the 47-case pilot (preliminary labels; not needed for judging).

**Pinned for the team:** code commit `REPO_REF`, judge `Qwen/Qwen3-4B-Instruct-2507` at a fixed revision, `float16`.
Change only `RESULTS_PATH`, `VARIANTS`, `OWNER` and `MODE`; every variant must be judged with identical settings.
Requirements: a GPU runtime (a T4 works; about 3 hours per variant). No Hugging Face token is needed for Qwen3.
Inputs are never modified. Outputs go to `OUTPUT_DIR/<run-name>/<variant>/` on Drive; see `med_copilot/docs/plan_judging_v2.md`.

In [ ]:
# ===================== CONFIG: edit only this cell =====================
MODE = "dry_run"                 # "dry_run" | "validate" | "judge"
REPO_URL = "https://github.com/ttogom/synthea-prep.git"   # public repository
REPO_REF = "f76d4b433b3b221ad7c4a24d98a69964397714a5"   # pinned code commit: do not change
DRIVE_ROOT = "/content/drive/MyDrive/synthea-prep"

# judge mode inputs (base or agentic run_experiments.py results.jsonl, read-only)
RESULTS_PATH = f"{DRIVE_ROOT}/runs/base/results.jsonl"   # agentic: f"{DRIVE_ROOT}/runs/agentic/results.jsonl"
PATIENTS_PATH = "experiments/input/test_so.jsonl"       # relative paths are inside the cloned repo
LABELS_PATH = "experiments/input/test_labels.jsonl"
VARIANTS = []                    # your assigned variant, e.g. ["llm_only"]
GENERATOR_MODELS = None          # e.g. ["gpt_4o_mini"]; None judges every final model
OUTPUT_DIR = f"{DRIVE_ROOT}/plan_judge_v2"
RUN_NAME = None                  # None: <run directory>__<results sha12>
OWNER = ""                       # your name, shown to teammates who hit your lease
WRITE_CSV = True
LIMIT = None                     # e.g. 20 for a smoke test; None judges everything pending
SAMPLE = None                    # benchmark: fixed pseudo-random N pending Plans (not with LIMIT)
TAKEOVER_STALE = False           # only after confirming nobody else is judging the variant
COUNT_TOKENS = True              # dry_run only: count every prompt with the judge tokenizer (no weights)

# judge model (selected 2026-11-21 after the pilot): keep these fixed
BACKEND = "vllm"                 # "vllm" (guided JSON) or "transformers" (fallback)
JUDGE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
JUDGE_REVISION = "cdbee75f17c01a7cc42f958dc650907174af0554"
DTYPE = "float16"                # same for every GPU, so all variants are judged identically
MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.9
CHAT_TEMPLATE_KWARGS = {}        # keep {} for Qwen3-4B-Instruct-2507 (non-thinking only)
ENGINE_ARGS = {}
BATCH_SIZE = 64                  # Plans per checkpointed batch
MAX_ATTEMPTS = 3

# validate mode (pilot model comparison)
PILOT_CASES = "experiments/validation/plan_judge_v2_pilot_cases.jsonl"
PILOT_REPORTS = f"{DRIVE_ROOT}/plan_judge_v2_pilot"
PILOT_ALLOW_DRAFT = False        # True only for a dry pipeline check before labels are reviewed
CANDIDATES = [
    {"model": "Qwen/Qwen3-4B-Instruct-2507", "revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
     "dtype": "float16"},
]
# =======================================================================


## Setup (clone, install, GPU check)

In [ ]:
import os, subprocess, sys
from pathlib import Path

if Path(DRIVE_ROOT).as_posix().startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

REPO = Path("/content/synthea-prep")
if not REPO.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(REPO)], check=True,
                   env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})   # LFS objects are not needed
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], check=True, capture_output=True,
                        text=True).stdout.strip()
print("Repository commit:", COMMIT)
if MODE != "dry_run":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-colab-judge.txt")],
                   check=True)
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
    print("GPU:", gpu.stdout.strip() or "none")
    if "T4" in gpu.stdout and DTYPE == "auto":
        print("WARNING: T4 GPUs lack bfloat16; set DTYPE = 'float16' in the config cell.")


## Credentials from Colab Secrets

In [ ]:
# HF_TOKEN comes from Colab Secrets and is passed only to the child process environment.
CHILD_ENV = dict(os.environ)
if MODE != "dry_run":
    try:
        from google.colab import userdata
        CHILD_ENV["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("HF_TOKEN loaded from Colab Secrets.")
    except Exception as exc:   # SecretNotFoundError, NotebookAccessError, or not on Colab
        print(f"No HF_TOKEN secret available ({type(exc).__name__}); only public, ungated models will load.")


## Run

In [ ]:
sys.path.insert(0, str(REPO))
from scripts.plan_judge_v2.colab import judge_argv, validate_argv

def repo_path(path):
    return str(path) if Path(path).is_absolute() else str(REPO / path)

def run(argv):
    print("$ python judge_plans_v2.py", " ".join(argv))
    process = subprocess.Popen([sys.executable, "judge_plans_v2.py", *argv], cwd=REPO, env=CHILD_ENV,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    code = process.wait()
    print(f"exit status {code}")
    return code

if MODE in ("dry_run", "judge"):
    if not VARIANTS:
        raise ValueError('Set VARIANTS to your assigned variant, e.g. ["llm_only"]')
    if MODE == "judge" and not OWNER.strip():
        raise ValueError("Set OWNER to your name so teammates can see who holds the variant")
    config = {"RESULTS_PATH": repo_path(RESULTS_PATH), "PATIENTS_PATH": repo_path(PATIENTS_PATH),
              "LABELS_PATH": repo_path(LABELS_PATH), "VARIANTS": VARIANTS, "GENERATOR_MODELS": GENERATOR_MODELS,
              "OUTPUT_DIR": OUTPUT_DIR, "RUN_NAME": RUN_NAME, "BACKEND": BACKEND, "JUDGE_MODEL": JUDGE_MODEL,
              "JUDGE_REVISION": JUDGE_REVISION, "DTYPE": DTYPE, "MAX_MODEL_LEN": MAX_MODEL_LEN,
              "GPU_MEMORY_UTILIZATION": GPU_MEMORY_UTILIZATION, "CHAT_TEMPLATE_KWARGS": CHAT_TEMPLATE_KWARGS,
              "ENGINE_ARGS": ENGINE_ARGS, "BATCH_SIZE": BATCH_SIZE, "MAX_ATTEMPTS": MAX_ATTEMPTS, "LIMIT": LIMIT,
              "SAMPLE": SAMPLE,
              "WRITE_CSV": WRITE_CSV, "OWNER": OWNER, "TAKEOVER_STALE": TAKEOVER_STALE,
              "DRY_RUN": MODE == "dry_run", "COUNT_TOKENS": COUNT_TOKENS}
    EXIT = run(judge_argv(config))
elif MODE == "validate":
    # One process per candidate so GPU memory is released between models.
    codes = [run(validate_argv(candidate, cases=repo_path(PILOT_CASES), out_dir=PILOT_REPORTS, backend=BACKEND,
                               max_model_len=MAX_MODEL_LEN, allow_draft=PILOT_ALLOW_DRAFT))
             for candidate in CANDIDATES]
    EXIT = max(codes) if codes else 1
    if any(code == 0 for code in codes):
        run(["compare", "--reports", PILOT_REPORTS])
else:
    raise ValueError(f"Unknown MODE {MODE}")


## Summary

In [ ]:
import json
if MODE == "judge":
    roots = sorted(Path(OUTPUT_DIR).glob(f"{RUN_NAME or '*'}/*/manifest.json"))
    for path in roots:
        manifest = json.loads(path.read_text())
        print(f"{path.parent.name}: run {manifest['run_id'][:8]}, judge {manifest['judge']['model_id']}"
              f"@{(manifest['judge']['revision_resolved'] or '')[:10]}, counts {manifest.get('counts')}")
        session = manifest["sessions"][-1] if manifest["sessions"] else {}
        print("  last session timing:", json.dumps(session.get("timing"), indent=2))
elif MODE == "validate" and (Path(PILOT_REPORTS) / "comparison.csv").exists():
    print((Path(PILOT_REPORTS) / "comparison.csv").read_text())
